# Adar1 cloud data access

This notebook provides information about how to download data from the [MalariaGEN Vector Observatory Anopheles darlingi Genomic Surveillance Project](https://www.malariagen.net/project/anopheles-darlingi-genomic-surveillance-project), for *Anopheles darlingi* via Google Cloud. This includes sample metadata, raw sequence reads, sequence read alignments, and single nucleotide polymorphism (SNP) calls. 

This notebook illustrates how to read data directly from the cloud, without having to first download any data locally. This notebook can be run from any computer, but will work best when run from a compute node within Google Cloud, because it will be physically closer to the data and so data transfer is faster. For example, this notebook can be run via [Google Colab](https://colab.research.google.com/) which are free interactive computing service running in the cloud.

To launch this notebook in the cloud and run it for yourself, click the launch icon (<i class="fas fa-rocket"></i>) at the top of the page and select one of the cloud computing services available.

## Data hosting

All data required for this notebook is hosted on Google Cloud Storage (GCS). Data are hosted in the `vo_adar_release_master_us_central1` bucket, which is a single-region bucket located in the United States. All data hosted in GCS are publicly accessible and do not require any authentication to access. 

## Setup

Running this notebook requires some Python packages to be installed:

In [1]:
%pip install -q malariagen_data

To make accessing these data more convenient, we've created the [malariagen_data](https://github.com/malariagen/malariagen-data-python) Python package. This is experimental so please let us know if you find any bugs or have any suggestions. See the [Adar1.0 API docs](https://malariagen.github.io/malariagen-data-python/latest/Adar1.0.html) for documentation of all functions available from this package. 

Import other packages we'll need to use here.

In [2]:
import numpy as np
import dask
import dask.array as da
from dask.diagnostics.progress import ProgressBar
# silence some warnings
dask.config.set(**{'array.slicing.split_large_chunks': False})
import allel
import malariagen_data

`Adar1` data access from Google Cloud is set up with the following code:

In [3]:
adar1 = malariagen_data.Adar1()
adar1

<MalariaGEN Adar1 API client>
Storage URL                           : gs://vo_adar_release_master_us_central1
Data releases available               : 1.0
Results cache                         : None
Cohorts analysis                      : 20260303
Site filters analysis                 : dt_20260301
Software version                      : malariagen_data 15.6.0.post407+1430baf
Client location                       : Iowa, United States (Google Cloud us-central1)
Data filtered to unrestricted use only: False
Data filtered to surveillance use only: False
Relevant data releases                : 1.0
---
Please note that data are subject to terms of use,
for more information see https://www.malariagen.net/data
or contact support@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v15.6.0.post407+1430baf/Adir1.html

## Sample sets

Data are organised into different releases. As an example, data in Adar1.0 are organised into 6 sample sets. Each of these sample sets corresponds to a set of mosquito specimens contributed by a collaborating study. Depending on your objectives, you may want to access data from only specific sample sets, or all sample sets.

To see which sample sets are available, load the sample set manifest into a pandas dataframe:

In [4]:
df_sample_sets = adar1.sample_sets(release="1.0")
df_sample_sets

,sample_set,sample_count,study_id,study_url,terms_of_use_expiry_date,terms_of_use_url,release,unrestricted_use
0,1357-VO-BR-SALLUM-VMF00326,272,1357-VO-BR-SALLUM,https://www.malariagen.net/partner_study/1357-...,2027-11-30,https://www.malariagen.net/data/our-approach-s...,1.0,False
1,1358-VO-GF-GENDRIN-VMF00327,139,1358-VO-GF-GENDRIN,https://www.malariagen.net/partner_study/1358-...,2027-11-30,https://www.malariagen.net/data/our-approach-s...,1.0,False
2,1359-VO-GY-NILES-ROBIN-VMF00328,18,1359-VO-GY-NILES-ROBIN,https://www.malariagen.net/partner_study/1359-...,2027-11-30,https://www.malariagen.net/data/our-approach-s...,1.0,False
3,1360-VO-PE-GAMBOA-VMF00329,89,1360-VO-PE-GAMBOA,https://www.malariagen.net/partner_study/1360-...,2027-11-30,https://www.malariagen.net/data/our-approach-s...,1.0,False
4,1361-VO-VE-GRILLET-VMF00330,126,1361-VO-VE-GRILLET,https://www.malariagen.net/partner_study/1361-...,2027-11-30,https://www.malariagen.net/data/our-approach-s...,1.0,False
5,1362-VO-CO-QUINONES-VMF00331,449,1362-VO-CO-QUINONES,https://www.malariagen.net/partner_study/1362-...,2027-11-30,https://www.malariagen.net/data/our-approach-s...,1.0,False


For more information about these sample sets, you can read about each sample set from the URLs under the field `study_url`.

## Sample metadata

Data about the samples that were sequenced to generate this data resource are available, including the time and place of collection, the gender of the specimen, and our call regarding the species of the specimen. These are organised by sample set.

E.g., load sample metadata for all samples in the Adar1.0 release into a [pandas DataFrame](https://pandas.pydata.org/pandas-docs/stable/user_guide/dsintro.html#dataframe):

In [5]:
df_samples = adar1.sample_metadata(sample_sets="1.0")
df_samples

,sample_id,partner_sample_id,contributor,country,location,year,month,latitude,longitude,sex_call,...,admin1_name,admin1_iso,admin2_name,taxon,cohort_admin1_year,cohort_admin1_month,cohort_admin1_quarter,cohort_admin2_year,cohort_admin2_month,cohort_admin2_quarter
0,VMF00326-0001,Coari_AM252_1,Maria Anice Mureb Sallum,Brazil,Coari,2022,11,-4.137,-63.150,UKN,...,Amazonas,BR-AM,Coari,darlingi,BR-AM_darl_2022,BR-AM_darl_2022_11,BR-AM_darl_2022_Q4,BR-AM_Coari_darl_2022,BR-AM_Coari_darl_2022_11,BR-AM_Coari_darl_2022_Q4
1,VMF00326-0002,Coari_AM252_2,Maria Anice Mureb Sallum,Brazil,Coari,2022,11,-4.137,-63.150,UKN,...,Amazonas,BR-AM,Coari,darlingi,BR-AM_darl_2022,BR-AM_darl_2022_11,BR-AM_darl_2022_Q4,BR-AM_Coari_darl_2022,BR-AM_Coari_darl_2022_11,BR-AM_Coari_darl_2022_Q4
2,VMF00326-0003,Coari_AM252_3,Maria Anice Mureb Sallum,Brazil,Coari,2022,11,-4.137,-63.150,UKN,...,Amazonas,BR-AM,Coari,darlingi,BR-AM_darl_2022,BR-AM_darl_2022_11,BR-AM_darl_2022_Q4,BR-AM_Coari_darl_2022,BR-AM_Coari_darl_2022_11,BR-AM_Coari_darl_2022_Q4
3,VMF00326-0004,Coari_AM252_4,Maria Anice Mureb Sallum,Brazil,Coari,2022,11,-4.137,-63.150,UKN,...,Amazonas,BR-AM,Coari,darlingi,BR-AM_darl_2022,BR-AM_darl_2022_11,BR-AM_darl_2022_Q4,BR-AM_Coari_darl_2022,BR-AM_Coari_darl_2022_11,BR-AM_Coari_darl_2022_Q4
4,VMF00326-0005,Coari_AM252_5,Maria Anice Mureb Sallum,Brazil,Coari,2022,11,-4.137,-63.150,UKN,...,Amazonas,BR-AM,Coari,darlingi,BR-AM_darl_2022,BR-AM_darl_2022_11,BR-AM_darl_2022_Q4,BR-AM_Coari_darl_2022,BR-AM_Coari_darl_2022_11,BR-AM_Coari_darl_2022_Q4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1089,VMF00331-0445,Tagachi_HER0112286021,Martha Quiñones,Colombia,Tagachí,2022,4,6.221,-76.726,UKN,...,Chocó,CO-CHO,Quibdó,darlingi,CO-CHO_darl_2022,CO-CHO_darl_2022_04,CO-CHO_darl_2022_Q2,CO-CHO_Quibdó_darl_2022,CO-CHO_Quibdó_darl_2022_04,CO-CHO_Quibdó_darl_2022_Q2
1090,VMF00331-0446,Tagachi_HER0112286022,Martha Quiñones,Colombia,Tagachí,2022,4,6.221,-76.726,UKN,...,Chocó,CO-CHO,Quibdó,darlingi,CO-CHO_darl_2022,CO-CHO_darl_2022_04,CO-CHO_darl_2022_Q2,CO-CHO_Quibdó_darl_2022,CO-CHO_Quibdó_darl_2022_04,CO-CHO_Quibdó_darl_2022_Q2
1091,VMF00331-0447,Tagachi_HER0112286023,Martha Quiñones,Colombia,Tagachí,2022,4,6.221,-76.726,UKN,...,Chocó,CO-CHO,Quibdó,darlingi,CO-CHO_darl_2022,CO-CHO_darl_2022_04,CO-CHO_darl_2022_Q2,CO-CHO_Quibdó_darl_2022,CO-CHO_Quibdó_darl_2022_04,CO-CHO_Quibdó_darl_2022_Q2
1092,VMF00331-0448,Tagachi_HER0112286024,Martha Quiñones,Colombia,Tagachí,2022,4,6.221,-76.726,UKN,...,Chocó,CO-CHO,Quibdó,darlingi,CO-CHO_darl_2022,CO-CHO_darl_2022_04,CO-CHO_darl_2022_Q2,CO-CHO_Quibdó_darl_2022,CO-CHO_Quibdó_darl_2022_04,CO-CHO_Quibdó_darl_2022_Q2


The `sample_id` column gives the sample identifier used throughout all Adir1.0 analyses.

The `country`, `location`, `latitude` and `longitude` columns give the location where the specimen was collected.

The `year` and `month` columns give the approximate date when the specimen was collected.

The `sex_call` column gives the gender as determined from the sequence data.

[Pandas](https://pandas.pydata.org/) can be used to explore and query the sample metadata in various ways. E.g., here is a summary of the numbers of samples by species:

In [6]:
df_samples.groupby("taxon").size()

taxon
darlingi    1094
dtype: int64

## SNP calls

Data on SNP calls, including the SNP positions, alleles, site filters, and genotypes, can be accessed as an [xarray Dataset](http://xarray.pydata.org/en/stable/user-guide/data-structures.html#dataset).

E.g., access SNP calls for contig 2 for all samples in `Adar1.0`.

In [7]:
ds_snps = adar1.snp_calls(region="2", sample_sets="1.0")
ds_snps

<xarray.Dataset> Size: 569GB
Dimensions:                       (variants: 30593416, alleles: 4,
                                   samples: 1094, ploidy: 2)
Coordinates:
    variant_position              (variants) int32 122MB dask.array<chunksize=(524288,), meta=np.ndarray>
    variant_contig                (variants) uint8 31MB dask.array<chunksize=(524288,), meta=np.ndarray>
    sample_id                     (samples) <U36 158kB dask.array<chunksize=(273,), meta=np.ndarray>
Dimensions without coordinates: variants, alleles, samples, ploidy
Data variables:
    variant_allele                (variants, alleles) |S1 122MB dask.array<chunksize=(524288, 4), meta=np.ndarray>
    variant_filter_pass_darlingi  (variants) bool 31MB dask.array<chunksize=(524288,), meta=np.ndarray>
    call_genotype                 (variants, samples, ploidy) int8 67GB dask.array<chunksize=(300000, 50, 2), meta=np.ndarray>
    call_GQ                       (variants, samples) int8 33GB dask.array<chunksize=(300000, 50), meta=np.ndarray>
    call_MQ                       (variants, samples) float32 134GB dask.array<chunksize=(300000, 50), meta=np.ndarray>
    call_AD                       (variants, samples, alleles) int16 268GB dask.array<chunksize=(300000, 50, 4), meta=np.ndarray>
    call_genotype_mask            (variants, samples, ploidy) bool 67GB dask.array<chunksize=(300000, 50, 2), meta=np.ndarray>
Attributes:
    contigs:  ('2', '3', 'X')

The arrays within this dataset are backed by [Dask arrays](https://docs.dask.org/en/latest/array.html), and can be accessed as shown below.

### SNP sites and alleles

We have called SNP genotypes in all samples at all positions in the genome where the reference allele is not "N". Data on this set of genomic positions and alleles for a given chromosome (e.g., 2RL) can be accessed as [Dask arrays](https://docs.dask.org/en/latest/array.html) as follows.

In [8]:
pos = ds_snps["variant_position"].data
pos

dask.array<array, shape=(30593416,), dtype=int32, chunksize=(524288,), chunktype=numpy.ndarray>

In [9]:
alleles = ds_snps["variant_allele"].data
alleles

dask.array<rechunk-merge, shape=(30593416, 4), dtype=|S1, chunksize=(524288, 4), chunktype=numpy.ndarray>

Data can be loaded into memory as a [NumPy array](https://numpy.org/doc/stable/user/absolute_beginners.html) as shown in the following examples.

In [10]:
# read first 10 SNP positions into a numpy array
p = pos[:10].compute()
p

array([11, 12, 13, 28, 36, 43, 45, 53, 54, 55], dtype=int32)

In [11]:
# read first 10 SNP alleles into a numpy array
a = alleles[:10].compute()
a

array([[b'C', b'A', b'G', b'T'],
       [b'T', b'A', b'C', b'G'],
       [b'G', b'A', b'C', b'T'],
       [b'C', b'A', b'G', b'T'],
       [b'A', b'C', b'G', b'T'],
       [b'G', b'A', b'C', b'T'],
       [b'A', b'C', b'G', b'T'],
       [b'G', b'A', b'C', b'T'],
       [b'A', b'C', b'G', b'T'],
       [b'G', b'A', b'C', b'T']], dtype='|S1')

Here the first column contains the reference alleles, and the remaining columns contain the alternate alleles.

Note that a byte string data type is used here for efficiency. E.g., the Python code `b'T'` represents a byte string containing the letter "T", which here stands for the nucleotide thymine.

Note that we have chosen to genotype all samples at all sites in the genome, assuming all possible SNP alleles. Not all of these alternate alleles will actually have been observed in the `Adar1` samples. To determine which sites and alleles are segregating, an allele count can be performed over the samples you are interested in. See the example below. 

### Site filters

SNP calling is not always reliable, and we have created some site filters to allow excluding low quality SNPs. 

Each set of site filters provides a "filter_pass" Boolean mask for each chromosome arm, where True indicates that the site passed the filter and is accessible to high quality SNP calling.

The site filters data can be accessed as dask arrays as shown in the examples below. 

In [12]:
# access gamb_colu_arab site filters as a dask array
filter_pass = ds_snps['variant_filter_pass_darlingi'].data
filter_pass

dask.array<array, shape=(30593416,), dtype=bool, chunksize=(524288,), chunktype=numpy.ndarray>

In [13]:
# read filter values for first 10 SNPs (True means the site passes filters)
f = filter_pass[:10].compute()
f

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True])

### SNP genotypes

SNP genotypes for individual samples are available. Genotypes are stored as a three-dimensional array, where the first dimension corresponds to genomic positions, the second dimension is samples, and the third dimension is ploidy (2). Values are coded as integers, where -1 represents a missing value, 0 represents the reference allele, and 1, 2, and 3 represent alternate alleles.

SNP genotypes can be accessed as dask arrays as shown below.

In [14]:
gt = ds_snps["call_genotype"].data
gt

dask.array<concatenate, shape=(30593416, 1094, 2), dtype=int8, chunksize=(300000, 50, 2), chunktype=numpy.ndarray>

Note that the columns of this array (second dimension) match the rows in the sample metadata, if the same sample sets were loaded. I.e.:

In [15]:
df_samples = adar1.sample_metadata(sample_sets="1.0")
gt = ds_snps["call_genotype"].data
len(df_samples) == gt.shape[1]

True

You can use this correspondance to apply further subsetting operations to the genotypes by querying the sample metadata. E.g.:

In [16]:
loc_darlingi = df_samples.eval("taxon == 'darlingi'").values
print(f"found {np.count_nonzero(loc_darlingi)} darlingi samples")

found 1094 darlingi samples


In [17]:
ds_snps_darlingi = ds_snps.isel(samples=loc_darlingi)
ds_snps_darlingi

<xarray.Dataset> Size: 569GB
Dimensions:                       (variants: 30593416, alleles: 4,
                                   samples: 1094, ploidy: 2)
Coordinates:
    variant_position              (variants) int32 122MB dask.array<chunksize=(524288,), meta=np.ndarray>
    variant_contig                (variants) uint8 31MB dask.array<chunksize=(524288,), meta=np.ndarray>
    sample_id                     (samples) <U36 158kB dask.array<chunksize=(182,), meta=np.ndarray>
Dimensions without coordinates: variants, alleles, samples, ploidy
Data variables:
    variant_allele                (variants, alleles) |S1 122MB dask.array<chunksize=(524288, 4), meta=np.ndarray>
    variant_filter_pass_darlingi  (variants) bool 31MB dask.array<chunksize=(524288,), meta=np.ndarray>
    call_genotype                 (variants, samples, ploidy) int8 67GB dask.array<chunksize=(300000, 45, 2), meta=np.ndarray>
    call_GQ                       (variants, samples) int8 33GB dask.array<chunksize=(300000, 45), meta=np.ndarray>
    call_MQ                       (variants, samples) float32 134GB dask.array<chunksize=(300000, 45), meta=np.ndarray>
    call_AD                       (variants, samples, alleles) int16 268GB dask.array<chunksize=(300000, 45, 4), meta=np.ndarray>
    call_genotype_mask            (variants, samples, ploidy) bool 67GB dask.array<chunksize=(300000, 45, 2), meta=np.ndarray>
Attributes:
    contigs:  ('2', '3', 'X')

Data can be read into memory as numpy arrays, e.g., read genotypes for the first 5 SNPs and the first 3 samples:

In [18]:
g = gt[:5, :3, :].compute()
g

array([[[0, 0],
        [0, 0],
        [0, 0]],

       [[0, 0],
        [0, 0],
        [0, 0]],

       [[0, 0],
        [0, 0],
        [0, 0]],

       [[0, 0],
        [0, 0],
        [0, 0]],

       [[0, 0],
        [0, 0],
        [0, 0]]], dtype=int8)

If you want to work with the genotype calls, you may find it convenient to use [scikit-allel](http://scikit-allel.readthedocs.org/).
E.g., the code below sets up a genotype array.

In [19]:
# use the scikit-allel wrapper class for genotype calls
gt = allel.GenotypeDaskArray(ds_snps["call_genotype"].data)
gt

<GenotypeDaskArray shape=(30593416, 1094, 2) dtype=int8>

## Example computation

Here's an example computation to count the number of segregating SNPs on the longest contig (???) that also pass site filters. This may take a minute or two, because it is scanning genotype calls at millions of SNPs in hundreds of samples.

In [20]:
# choose contig (longest contig)
region = "2"
# choose site filter mask

# choose sample sets
sample_sets = ["1357-VO-BR-SALLUM-VMF00326"]

# access SNP calls
ds_snps = adar1.snp_calls(region=region, sample_sets=sample_sets)

# locate pass sites
loc_pass = ds_snps[f"variant_filter_pass_darlingi"].values

# perform an allele count over genotypes
gt = allel.GenotypeDaskArray(ds_snps["call_genotype"].data)
with ProgressBar():
    ac = gt.count_alleles(max_allele=3).compute()

# locate segregating sites
loc_seg = ac.is_segregating()

# count segregating and pass sites
n_pass_seg = np.count_nonzero(loc_pass & loc_seg)
n_pass_seg

[########################################] | 100% Completed | 6.29 sms


22073253

## Running larger computations

Please note that free cloud computing services such as Google Colab provide only limited computing resources. Thus although these services are able to efficiently read `Adar1` data stored on Google Cloud, you may find that you run out of memory, or computations take a long time running on a single core. If you would like any suggestions regarding how to set up more powerful compute resources in the cloud, please feel free to get in touch via the [malariagen/vector-data GitHub discussion board](https://github.com/malariagen/vector-data/discussions).

## Feedback and suggestions

If there are particular analyses you would like to run, or if you have other suggestions for useful documentation we could add to this site, we would love to know, please get in touch via the [malariagen/vector-data GitHub discussion board](https://github.com/malariagen/vector-data/discussions).